# Penyetelan XGBoost dengan Random Search

Notebook ini menguji apakah penyetelan membuat XGBoost jelas lebih baik dari parameter default. Data uji tidak dipakai.

**Aturan keputusan:** hasil penyetelan hanya dipakai kalau PR-AUC-nya lebih tinggi dan rentangnya tidak bertumpuk dengan default pada fold yang sama. Kalau tidak, default tetap dipakai.

**Keterbatasan:** Analisis mengikuti dokumentasi UCI yang menyatakan tiap baris mewakili satu pelanggan. Audit menemukan pola berulang per 50 baris, sehingga independensi antarbaris tidak terjamin dan skor model bisa terlalu optimistis. Skor CV acak tidak boleh dibaca sebagai performa pada pelanggan baru.

## 1. Persiapan

In [1]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
from scipy.stats import loguniform, randint, uniform
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import (
    RandomizedSearchCV, RepeatedStratifiedKFold, StratifiedGroupKFold, StratifiedKFold, train_test_split
)
from xgboost import XGBClassifier

SEED = 180915
N_SPLITS = 5
N_REPEATS = 10
N_ITER = 60

# Path relatif terhadap root project
ROOT = Path.cwd().parent if Path.cwd().name == 'program' else Path.cwd()

In [2]:
df = pd.read_csv(ROOT / 'dataset_clean' / 'customer_churn_clean.csv')
groups = pd.read_csv(ROOT / 'dataset_clean' / 'validation_groups.csv')['audit_group']
assert len(df) == len(groups)

TARGET = 'Churn'
FEATURES_ALL = [col for col in df.columns if col not in (TARGET, 'Age_Binning')]
FEATURES_REDUCED = [col for col in FEATURES_ALL if col not in ('Status', 'Complains')]
FEATURE_SETS = {'12 fitur': FEATURES_ALL, '10 fitur': FEATURES_REDUCED}
assert 'audit_group' not in FEATURES_ALL and TARGET not in FEATURES_ALL
assert len(FEATURES_ALL) == 12 and len(FEATURES_REDUCED) == 10

# Pembagian yang sama dengan notebook perbandingan model. Baris uji tidak dipakai
train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, stratify=df[TARGET], random_state=SEED
)
assert not set(train_idx) & set(test_idx)
train = df.loc[train_idx].reset_index(drop=True)
y_train = train[TARGET]
groups_train = groups.loc[train_idx].reset_index(drop=True)
assert len(train) == 2280 and y_train.sum() == 357
assert len(test_idx) == 570 and df.loc[test_idx, TARGET].sum() == 89

print(f'Latih: {len(train):,} baris, {y_train.sum():,} churn')
print(f'Uji (tidak dipakai): {len(test_idx):,} baris')

Latih: 2,280 baris, 357 churn
Uji (tidak dipakai): 570 baris


## 2. Pencarian

Random search mencoba 60 kombinasi pada 12 fitur, dengan PR-AUC sebagai metrik. Rentangnya menyertakan nilai yang konservatif, seperti pohon dangkal dan `min_child_weight` besar.

In [3]:
param_distributions = {
    'n_estimators': randint(100, 601),
    'max_depth': randint(2, 9),
    'learning_rate': loguniform(0.01, 0.3),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'min_child_weight': randint(1, 11),
}

search = RandomizedSearchCV(
    XGBClassifier(random_state=SEED, eval_metric='logloss'),
    param_distributions=param_distributions,
    n_iter=N_ITER,
    scoring='average_precision',
    cv=StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED),
    random_state=SEED,
    n_jobs=1,  # XGBoost sudah memakai banyak inti
    refit=False,
)

started = time.time()
search.fit(train[FEATURES_ALL], y_train)
assert len(search.cv_results_['params']) == N_ITER

print(f'Selesai dalam {(time.time() - started) / 60:.1f} menit')
print(f"Kombinasi dicoba: {len(search.cv_results_['params'])}")
print(f'PR-AUC terbaik saat pencarian: {search.best_score_:.3f} (optimistis, karena dipakai untuk memilih)')
print('Parameter terbaik:')
for name, value in search.best_params_.items():
    print(f'  {name}: {value:.4f}' if isinstance(value, float) else f'  {name}: {value}')

Selesai dalam 1.4 menit
Kombinasi dicoba: 60
PR-AUC terbaik saat pencarian: 0.941 (optimistis, karena dipakai untuk memilih)
Parameter terbaik:
  colsample_bytree: 0.6994
  learning_rate: 0.1127
  max_depth: 8
  min_child_weight: 2
  n_estimators: 180
  subsample: 0.9807


## 3. Penilaian ulang

Default dan kombinasi terbaik dinilai pada fold yang sama, dengan seed yang berbeda dari pencarian. Skor dihitung per fold lalu dirata-ratakan per ulangan, dan fold tanpa churn dilewati. Rentang min–maks menggambarkan variasi antarulangan, bukan interval kepercayaan. Cek grup hanya catatan internal.

In [4]:
def random_cv_repeats(X, y):
    splits = list(RepeatedStratifiedKFold(
        n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED + 1
    ).split(X, y))
    return [splits[i:i + N_SPLITS] for i in range(0, len(splits), N_SPLITS)]


def group_cv_repeats(X, y, group_values):
    repeats = []
    for repeat in range(N_REPEATS):
        splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED + repeat)
        splits = list(splitter.split(X, y, group_values))
        for fit_idx, val_idx in splits:
            assert not set(group_values.iloc[fit_idx]) & set(group_values.iloc[val_idx])
        repeats.append(splits)
    return repeats


def evaluate(params, X, y, repeats):
    rows = []
    for repeat, splits in enumerate(repeats):
        fold_pr, fold_roc = [], []
        for fit_idx, val_idx in splits:
            y_val = y.iloc[val_idx]
            if y_val.nunique() < 2:
                continue
            model = XGBClassifier(random_state=SEED, eval_metric='logloss', **params)
            model.fit(X.iloc[fit_idx], y.iloc[fit_idx])
            scores = model.predict_proba(X.iloc[val_idx])[:, 1]
            fold_pr.append(average_precision_score(y_val, scores))
            fold_roc.append(roc_auc_score(y_val, scores))
        rows.append({'repeat': repeat, 'pr_auc': np.mean(fold_pr), 'roc_auc': np.mean(fold_roc)})
    return rows


CV_REPEATS = {
    'CV acak': random_cv_repeats(train, y_train),
    'Cek grup': group_cv_repeats(train, y_train, groups_train),
}
PARAM_SETS = {'Default': {}, 'Disetel': search.best_params_}

started = time.time()
records = []
for feature_set, features in FEATURE_SETS.items():
    for cv_name, repeats in CV_REPEATS.items():
        for param_name, params in PARAM_SETS.items():
            for row in evaluate(params, train[features], y_train, repeats):
                records.append({'fitur': feature_set, 'cv': cv_name, 'parameter': param_name, **row})
results = pd.DataFrame(records)
print(f'Selesai dalam {(time.time() - started) / 60:.1f} menit')

Selesai dalam 1.1 menit


In [5]:
def fmt(values):
    text = f'{values.mean():.3f} ({values.min():.3f}–{values.max():.3f})'
    return text.replace('.', ',')


table = results.groupby(['fitur', 'cv', 'parameter']).agg(
    pr_auc=('pr_auc', fmt), roc_auc=('roc_auc', fmt)
)
table = table.rename(columns={'pr_auc': 'PR-AUC', 'roc_auc': 'ROC-AUC'})
print(table.to_string())

                                          PR-AUC              ROC-AUC
fitur    cv       parameter                                          
10 fitur CV acak  Default    0,885 (0,875–0,895)  0,969 (0,965–0,972)
                  Disetel    0,886 (0,877–0,894)  0,970 (0,967–0,972)
         Cek grup Default    0,231 (0,171–0,294)  0,622 (0,481–0,720)
                  Disetel    0,245 (0,180–0,357)  0,644 (0,481–0,724)
12 fitur CV acak  Default    0,934 (0,926–0,939)  0,980 (0,977–0,983)
                  Disetel    0,935 (0,929–0,942)  0,981 (0,979–0,984)
         Cek grup Default    0,474 (0,323–0,588)  0,802 (0,754–0,858)
                  Disetel    0,491 (0,307–0,598)  0,811 (0,764–0,857)


## 4. Kesimpulan menurut aturan keputusan

In [6]:
main = results[(results['fitur'] == '12 fitur') & (results['cv'] == 'CV acak')]
default_scores = main[main['parameter'] == 'Default'].set_index('repeat')['pr_auc']
tuned_scores = main[main['parameter'] == 'Disetel'].set_index('repeat')['pr_auc']

is_clearly_better = (tuned_scores.mean() > default_scores.mean()) and (tuned_scores.min() > default_scores.max())
paired_difference = tuned_scores - default_scores

print(f'Default: {fmt(default_scores)}')
print(f'Disetel: {fmt(tuned_scores)}')
print(f'Selisih per ulangan: rata-rata {paired_difference.mean():+.4f}, '
      f'min {paired_difference.min():+.4f}, maks {paired_difference.max():+.4f}')
print('Jelas lebih baik:', 'ya' if is_clearly_better else 'tidak')
print('Parameter yang dipakai menurut aturan:', 'Disetel' if is_clearly_better else 'Default')

Default: 0,934 (0,926–0,939)
Disetel: 0,935 (0,929–0,942)
Selisih per ulangan: rata-rata +0.0015, min -0.0012, maks +0.0044
Jelas lebih baik: tidak
Parameter yang dipakai menurut aturan: Default


## Cara membaca hasil

- Skor saat pencarian lebih optimistis, karena dipakai untuk memilih kombinasi.
- Penilaian ulang memakai data latih yang sama, jadi data uji tetap menjadi pemeriksaan akhir.
- Skor tinggi pada CV acak dibaca dengan curiga, dan tidak boleh disebut performa pada pelanggan baru.
- Model belum dikunci di notebook ini.